In [20]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torch
import torch.nn as nn
import numpy as np 
import os
import matplotlib.pyplot as plt

from vae import VAE_model, VAE_loss
from dataset import FilteredMNIST

# Download and preprocess data

In [21]:
# MNIST Dataset transformation
mnist_transform = transforms.Compose([
    transforms.ToTensor(), # Converts to [0, 1] interval
    transforms.Lambda(lambda x: torch.flatten(x)) # Flattens the image to a 1D vector
])

# Download and load the MNIST dataset
full_train_dataset = datasets.MNIST(root='mnist_data', train=True, transform=mnist_transform, download=True)
print("Original size of the dataset: ", len(full_train_dataset))
filtered_labels = [0, 1, 7]
print("Filtered labels: ", filtered_labels)
train_dataset = FilteredMNIST(full_train_dataset, filtered_labels)
print("New size of the dataset: ", len(train_dataset))

batch_size = 256
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True,num_workers=4)

Original size of the dataset:  60000
Filtered labels:  [0, 1, 7]
New size of the dataset:  18930


# Load model

In [22]:
device = torch.device("cpu")
N_train, D = len(train_dataset), 28*28 # MNIST images are 28x28 pixels

# Parameters of the VAE
d = 2  # The latent space dimension
H = 32
lambda_reg = 1e-7  # For the weights of the networks
epochs = 500
warmup = int(0.33 * epochs)
learning_rate = 1e-3
clipping_value = 1
batch_size = 128

In [23]:
# The model and the optimizer for the VAE
model = VAE_model(d=d, D=D, H=H, activFun=nn.Tanh()).to(device)
optimizer_model = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=lambda_reg)

In [25]:
ELBO = np.zeros((epochs, 1))

for epoch in range(epochs):

    train_loss = 0
    train_loss_num = 0

    model.train()

    # Train for all batches
    for X_batch, _ in train_loader:

        X_batch = X_batch.to(device)

        # Forward pass
        MU_X_eval, LOG_VAR_X_eval, Z_ENC_eval, MU_Z_eval, LOG_VAR_Z_eval = model(X_batch)

        # Warmup annealing
        if warmup == 0:
            r = 0
        else:
            r = min(1.0, epoch / warmup)

        # Loss
        loss = VAE_loss(
            x=X_batch,
            mu_x=MU_X_eval,
            log_var_x=LOG_VAR_X_eval,
            mu_z=MU_Z_eval,
            log_var_z=LOG_VAR_Z_eval,
            r=r
        )

        optimizer_model.zero_grad()
        loss.backward()
        optimizer_model.step()

        train_loss += loss.item()
        train_loss_num += 1

    ELBO[epoch] = train_loss / train_loss_num

    if epoch % 20 == 0:
        print(f"[Epoch: {epoch}/{epochs}] [objective: {ELBO[epoch,0]:.3f}]")

print("[ELBO train:", round(ELBO[epochs-1,0],2), "]")
print("Training finished")

[Epoch: 0/500] [objective: -1070.459]


KeyboardInterrupt: 

In [ ]:
# Plot ELBO loss
plt.figure(figsize=(6,4))
plt.plot(ELBO)
plt.xlabel("Epoch")
plt.ylabel("ELBO loss")
plt.title("Training Loss Curve")
plt.grid()

plt.tight_layout()
plt.show()


# Save model
save_dir = "saved_models"
os.makedirs(save_dir, exist_ok=True)

model_path = os.path.join(save_dir, "vae_model_mnist.pt")

torch.save(model.state_dict(), model_path)

print(f"Model saved in {model_path}")

In [ ]:
# if model already exists, load it
model_path = "saved_models/vae_model_mnist.pt"
if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path))
    print(f"Model loaded from {model_path}")